# Búsqueda de hiperparámetros de LightGBM con Telemetry y Cassette

**Resultado de esta ejecución.** En las 8 personas reservadas para prueba, el DART actual obtuvo un F1 macro de **0,7981** y el GBDT elegido por validación interna obtuvo **0,7968**. La búsqueda no mejoró el F1 macro global en esta partición. La sección 4 muestra los resultados por cohorte. La sección 5 guarda el candidato experimental entrenado con las 56 noches.

¿Mejora el F1 macro de las cinco etapas al cambiar el número de árboles, la regularización o el tipo de boosting? El control es el LightGBM mixto del notebook de Jhoan. Usamos todos los registros emparejados disponibles y el mismo preprocesamiento multicanal. Se puede ejecutar desde la raíz o desde `Notebooks/experiments/` con el entorno del proyecto y los datos recuperados mediante `dvc pull`.

El experimento conserva las personas de prueba del modelo mixto original. Dentro del resto del dataset compara configuraciones en tres particiones por persona, equilibradas por cohorte. Las configuraciones y el criterio de selección se fijan antes de evaluar el conjunto de prueba. Es una evaluación de desarrollo sobre un dataset ya explorado.

In [1]:
import hashlib
import json
import os
import sys
import time
from pathlib import Path
from tempfile import NamedTemporaryFile, gettempdir

import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.dvc/config').is_file())
sys.path.insert(0, str(ROOT / 'packages/sleep-staging/src'))
from sleep_staging import PreprocessingConfig, PreprocessingPipeline
from sleep_staging.datasets import VALID_STAGES, discover_sleep_cassette_records, discover_sleep_telemetry_records
from sleep_staging.training import load_or_build_supervised_dataset

SEED, THREADS = 42, 4
STAGES = list(VALID_STAGES)
DATA = ROOT / 'datos'
reference = DATA / 'experimentos/lightgbm_st_sc'
spec = json.loads((reference / 'experimento.json').read_text('utf-8'))
columns = spec['feature_columns']
sc_cache = Path(gettempdir()) / 'sleep-edfx-sc-features-v1'
st_cache = DATA / 'processed/supervised'

## 1. Entradas preparadas y población

Telemetry usa la caché compartida de DVC. Para Cassette se reutiliza la caché por noche del notebook original, que se puede regenerar a partir de los EDF de DVC. Las columnas de EMG sin señal equivalente en Cassette se dejan ausentes para LightGBM.

In [2]:
def cargar(registros, cohorte, cache):
    cfg = spec['preprocessing_telemetry' if cohorte == 'Telemetry' else 'preprocessing_cassette']
    ds = load_or_build_supervised_dataset(registros, PreprocessingPipeline(PreprocessingConfig(**cfg)), cache)
    esperadas = columns if cohorte == 'Telemetry' else [c for c in columns if not c.startswith('EMG submental_')]
    assert set(ds.features) == set(esperadas) and ds.features.columns.is_unique
    frame = ds.features.astype('float32').reindex(columns=columns)
    etiquetas = ds.metadata[['record_id','subject_id','epoch_index']].copy()
    etiquetas['stage'] = ds.labels.to_numpy()
    etiquetas['cohorte'] = cohorte
    return frame, etiquetas

st = discover_sleep_telemetry_records(DATA / 'sleep-telemetry')
sc = discover_sleep_cassette_records(DATA / 'sleep-cassette')
partes = [cargar(st, 'Telemetry', st_cache)]
partes.extend(cargar([r], 'Cassette', sc_cache) for r in sc)
X = pd.concat([x for x, _ in partes], ignore_index=True)
meta = pd.concat([m for _, m in partes], ignore_index=True)
del partes
y = meta.stage
assert len(X) == len(meta) and set(y) == set(STAGES)
assert not meta.duplicated(['record_id','epoch_index']).any()
desarrollo, prueba = set(spec['train_subjects']), set(spec['validation_subjects'])
assert desarrollo.isdisjoint(prueba) and desarrollo | prueba == set(meta.subject_id)
meta['uso'] = np.where(meta.subject_id.isin(prueba), 'Prueba reservada', 'Desarrollo')
display(meta.groupby(['uso','cohorte']).agg(personas=('subject_id','nunique'), noches=('record_id','nunique'), segmentos=('stage','size')))
print(f'{len(meta):,} segmentos de 30 s y {X.shape[1]} características.')

Cargando dataset desde caché: E:\PROJECTS\Maestria Inteligencia Artificial\masterIADesarollo\micro-proyecto-grupo-7\data\processed\supervised\18c3c51b05e3


Cargando dataset desde caché: C:\Users\sebas\AppData\Local\Temp\sleep-edfx-sc-features-v1\4879efb4ea1a


Cargando dataset desde caché: C:\Users\sebas\AppData\Local\Temp\sleep-edfx-sc-features-v1\fc044f37e663


Cargando dataset desde caché: C:\Users\sebas\AppData\Local\Temp\sleep-edfx-sc-features-v1\4aa9cd546c9e


Cargando dataset desde caché: C:\Users\sebas\AppData\Local\Temp\sleep-edfx-sc-features-v1\e6f9fd066d2a


Cargando dataset desde caché: C:\Users\sebas\AppData\Local\Temp\sleep-edfx-sc-features-v1\5234d481567e
Cargando dataset desde caché: C:\Users\sebas\AppData\Local\Temp\sleep-edfx-sc-features-v1\fb0a1478b297


... (3 outputs left out)

Cargando dataset desde caché: C:\Users\sebas\AppData\Local\Temp\sleep-edfx-sc-features-v1\4d594a9e41e2


C:\Users\sebas\AppData\Local\Temp\ipykernel_39536\154995976.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  X = pd.concat([x for x, _ in partes], ignore_index=True)


personas  noches  segmentos
uso              cohorte                               
Desarrollo       Cassette          9       9      24472
                 Telemetry        17      34      32762
Prueba reservada Cassette          3       3       8449
                 Telemetry         5      10       9929

75,612 segmentos de 30 s y 2280 características.


## 2. Configuraciones y particiones internas

La búsqueda acotada incluye el control DART, una variante DART más corta y dos variantes GBDT. Conservamos los pesos de las cinco clases y el preprocesamiento. Gana el mayor F1 macro medio de las tres particiones internas; un empate favorece la configuración que aparece antes en la lista. Cada persona de desarrollo se valida una vez. El conjunto de prueba queda reservado hasta el final.

In [3]:
base = dict(spec['parameters'], random_state=SEED, n_jobs=THREADS)
def parametros(cambios):
    params = {**base, **cambios}
    if params['boosting_type'] == 'gbdt':
        for key in ('drop_rate','skip_drop'):
            params.pop(key, None)
    return params

candidatos = {
    'DART actual': parametros({}),
    'DART regularizado': parametros(dict(n_estimators=350, learning_rate=0.05, num_leaves=28,
        min_child_samples=55, reg_lambda=3.0, max_bin=63, drop_rate=0.08)),
    'GBDT compacto': parametros(dict(boosting_type='gbdt', n_estimators=180, learning_rate=0.045,
        num_leaves=24, min_child_samples=60, reg_lambda=3.0, max_bin=63)),
    'GBDT profundo': parametros(dict(boosting_type='gbdt', n_estimators=280, learning_rate=0.035,
        num_leaves=48, min_child_samples=40, reg_lambda=2.0, max_bin=63)),
}
personas = meta.loc[meta.subject_id.isin(desarrollo), ['subject_id','cohorte']].drop_duplicates().sort_values(['cohorte','subject_id'])
assert personas.subject_id.is_unique
rondas = []
for a, b in StratifiedKFold(3, shuffle=True, random_state=SEED).split(personas, personas.cohorte):
    tr, va = set(personas.iloc[a].subject_id), set(personas.iloc[b].subject_id)
    assert tr.isdisjoint(va) and tr | va == desarrollo
    rondas.append((np.flatnonzero(meta.subject_id.isin(tr)), np.flatnonzero(meta.subject_id.isin(va))))
assert sorted(np.concatenate([v for _, v in rondas])) == list(meta.index[meta.subject_id.isin(desarrollo)])
display(pd.DataFrame([dict(ronda=i, entrenamiento=len(a), validación=len(b),
                           personas_validación=', '.join(sorted(meta.subject_id.iloc[b].unique())))
                      for i,(a,b) in enumerate(rondas,1)]))

# Una caché por experimento permite reanudar sin mezclar implementaciones o parámetros.
notebook = ROOT / 'Notebooks/experiments/busqueda_hiperparametros_lightgbm_mixto.ipynb'
codigo = [c['source'] for c in json.loads(notebook.read_text('utf-8'))['cells'] if c['cell_type']=='code']
huella = hashlib.sha256(json.dumps([codigo, lgb.__version__, sklearn.__version__]).encode())
for p in [*sorted((ROOT/'datos').glob('*.dvc')), reference/'experimento.json', *sorted((ROOT/'packages/sleep-staging/src/sleep_staging').rglob('*.py'))]:
    huella.update(p.read_bytes())
# Un alias exacto conserva la caché del mismo experimento tras mover el cuaderno.
aliases = json.loads(notebook.read_text('utf-8'))['metadata']['cache_tras_traslado']
OUT = ROOT / '.sleep-cv/ajuste-lightgbm' / aliases.get(huella.hexdigest(), huella.hexdigest()[:16])
OUT.mkdir(parents=True, exist_ok=True)
print('Resultados locales:', OUT)

def guardar_json(destino, datos):
    with NamedTemporaryFile('w', dir=destino.parent, delete=False, encoding='utf-8') as temp:
        json.dump(datos, temp, ensure_ascii=False, indent=2, allow_nan=False)
    os.replace(temp.name, destino)

def metricas(real, pred):
    return {'F1 macro': f1_score(real,pred,labels=STAGES,average='macro',zero_division=0),
            'Exactitud': accuracy_score(real,pred), 'Kappa': cohen_kappa_score(real,pred)}

,ronda,entrenamiento,validación,personas_validación
0,1,38163,19071,"S06, S11, S13, S14, S19, S20, SC410, SC426, SC435"
1,2,37236,19998,"S05, S07, S08, S16, S21, S24, SC417, SC441, SC452"
2,3,39069,18165,"S04, S09, S12, S18, S22, SC407, SC434, SC482"


Resultados locales: E:\PROJECTS\Maestria Inteligencia Artificial\masterIADesarollo\micro-proyecto-grupo-7\.sleep-cv\ajuste-lightgbm\8731d8caf3d24703


## 3. Búsqueda sobre los sujetos de desarrollo

Cada fila de la salida corresponde a un ajuste real. El tiempo incluye selección de filas, ajuste y predicción de validación. Las métricas de cada ajuste se guardan antes de iniciar el siguiente.

In [4]:
resultados = []
def evaluar_candidato(nombre):
    for i, (tr,va) in enumerate(rondas, 1):
        destino = OUT / f'{nombre}-{i}.json'
        if destino.exists():
            fila = json.loads(destino.read_text('utf-8'))
        else:
            inicio = time.perf_counter()
            modelo = lgb.LGBMClassifier(**candidatos[nombre])
            modelo.fit(X.iloc[tr], y.iloc[tr])
            fila = {'Configuración':nombre, 'Ronda':i, **metricas(y.iloc[va], modelo.predict(X.iloc[va])),
                    'Segundos':time.perf_counter()-inicio}
            guardar_json(destino, fila)
        print(f"{nombre} · ronda {i} · F1 macro {fila['F1 macro']:.4f}", flush=True)
        resultados.append(fila)

In [5]:
evaluar_candidato('GBDT compacto')

GBDT compacto · ronda 1 · F1 macro 0.7989


GBDT compacto · ronda 2 · F1 macro 0.7908


GBDT compacto · ronda 3 · F1 macro 0.7934


In [6]:
evaluar_candidato('GBDT profundo')

GBDT profundo · ronda 1 · F1 macro 0.8026


GBDT profundo · ronda 2 · F1 macro 0.7843


GBDT profundo · ronda 3 · F1 macro 0.7995


In [7]:
evaluar_candidato('DART regularizado')

DART regularizado · ronda 1 · F1 macro 0.7946


DART regularizado · ronda 2 · F1 macro 0.7889


DART regularizado · ronda 3 · F1 macro 0.7870


In [8]:
evaluar_candidato('DART actual')

DART actual · ronda 1 · F1 macro 0.7973


DART actual · ronda 2 · F1 macro 0.7881


DART actual · ronda 3 · F1 macro 0.7877


## 4. Selección y evaluación sobre los mismos sujetos de prueba

El control es el modelo mixto guardado en DVC. Si la búsqueda elige una alternativa, se reajusta con las 26 personas de desarrollo y se compara con el control en los mismos segmentos de las 8 personas reservadas. El F1 global de cinco rondas de otros experimentos corresponde a otra partición.

In [9]:
detalle = pd.DataFrame(resultados).drop_duplicates(['Configuración','Ronda'])
assert detalle.groupby('Configuración').size().reindex(candidatos).eq(len(rondas)).all()
ranking = detalle.groupby('Configuración')[['F1 macro','Exactitud','Kappa','Segundos']].mean().reindex(candidatos)
ranking['Desviación F1'] = detalle.groupby('Configuración')['F1 macro'].std()
ranking = ranking.sort_values('F1 macro', ascending=False, kind='stable')
nombre = ranking.index[0]
display(ranking.round(4))
print('Selección por F1 macro interno:', nombre)
print(json.dumps(candidatos[nombre], ensure_ascii=False, indent=2))

tr = meta.index[meta.subject_id.isin(desarrollo)]
te = meta.index[meta.subject_id.isin(prueba)]
control = lgb.Booster(model_file=str(reference/'model.txt'))
assert control.num_feature()==len(columns)
control_pred = np.array(spec['classes'])[control.predict(X.loc[te], num_threads=THREADS).argmax(axis=1)]
predicciones = {'DART actual (DVC)': control_pred}
if nombre != 'DART actual':
    candidato = lgb.LGBMClassifier(**candidatos[nombre]).fit(X.loc[tr], y.loc[tr])
    predicciones[nombre] = candidato.predict(X.loc[te])
    del candidato
filas = []
for modelo, pred in predicciones.items():
    for cohorte in ['Ambas', 'Telemetry', 'Cassette']:
        mask = np.ones(len(te),dtype=bool) if cohorte=='Ambas' else meta.loc[te,'cohorte'].eq(cohorte).to_numpy()
        filas.append({'Modelo':modelo,'Cohorte':cohorte,'Segmentos':int(mask.sum()),**metricas(y.loc[te].to_numpy()[mask],pred[mask])})
display(pd.DataFrame(filas).set_index(['Modelo','Cohorte']).round(4))
mejora = metricas(y.loc[te], predicciones.get(nombre, control_pred))['F1 macro'] - metricas(y.loc[te],control_pred)['F1 macro']
print(f'Diferencia de F1 macro en prueba frente al control: {mejora:+.4f}.')
guardar_json(OUT/'evaluacion.json', {'selección':nombre,'parámetros':candidatos[nombre], 'prueba':filas,'delta_f1':mejora})

,F1 macro,Exactitud,Kappa,Segundos,Desviación F1
Configuración,,,,,
GBDT profundo,0.7954,0.8689,0.8164,112.9126,0.0098
GBDT compacto,0.7944,0.8641,0.8106,67.6673,0.0042
DART actual,0.7910,0.8615,0.8069,474.5310,0.0055
DART regularizado,0.7902,0.8594,0.8046,136.9517,0.0040


Selección por F1 macro interno: GBDT profundo
{
  "boosting_type": "gbdt",
  "class_weight": {
    "N1": 2.2,
    "N2": 0.41,
    "N3": 1.56,
    "REM": 1.0,
    "W": 2.01
  },
  "colsample_bytree": 0.6,
  "importance_type": "split",
  "learning_rate": 0.035,
  "max_depth": 8,
  "min_child_samples": 40,
  "min_child_weight": 0.001,
  "min_split_gain": 0.0,
  "n_estimators": 280,
  "n_jobs": 4,
  "num_leaves": 48,
  "objective": "multiclass",
  "random_state": 42,
  "reg_alpha": 0.1,
  "reg_lambda": 2.0,
  "subsample": 0.8,
  "subsample_for_bin": 200000,
  "subsample_freq": 1,
  "verbose": -1,
  "max_bin": 63
}


Segmentos  F1 macro  Exactitud   Kappa
Modelo            Cohorte                                          
DART actual (DVC) Ambas          18378    0.7981     0.8504  0.7942
                  Telemetry       9929    0.7948     0.8181  0.7541
                  Cassette        8449    0.7686     0.8884  0.7608
GBDT profundo     Ambas          18378    0.7968     0.8556  0.8010
                  Telemetry       9929    0.7910     0.8164  0.7500
                  Cassette        8449    0.7721     0.9016  0.7937

Diferencia de F1 macro en prueba frente al control: -0.0013.


## 5. Ajuste final con todos los datos disponibles

Después de la evaluación, se entrena la configuración seleccionada con las 34 personas de ambas cohortes y se guarda en el directorio local del experimento. La tabla anterior describe el modelo ajustado solo con desarrollo. Las predicciones sobre este conjunto completo no sirven para estimar el rendimiento futuro del modelo final.

In [10]:
artefacto = OUT / 'modelo_todos_los_sujetos.txt'
if not artefacto.exists():
    final = lgb.LGBMClassifier(**candidatos[nombre]).fit(X, y)
    with NamedTemporaryFile(dir=OUT, delete=False) as tmp:
        provisional = Path(tmp.name)
    final.booster_.save_model(str(provisional))
    verificado = lgb.Booster(model_file=str(provisional))
    np.testing.assert_allclose(verificado.predict(X.iloc[:20], num_threads=THREADS), final.predict_proba(X.iloc[:20]))
    assert list(final.classes_) == spec['classes']
    os.replace(provisional, artefacto)
manifest = {**spec, 'parameters':candidatos[nombre], 'train_subjects':sorted(set(meta.subject_id)),
            'validation_subjects':[], 'selection':nombre, 'selection_evaluation':'evaluacion.json'}
guardar_json(OUT/'experimento.json',manifest)
print(f'Modelo final entrenado con {meta.subject_id.nunique()} personas, {meta.record_id.nunique()} noches y {len(meta)} segmentos.')
print(artefacto)

Modelo final entrenado con 34 personas, 56 noches y 75612 segmentos.
E:\PROJECTS\Maestria Inteligencia Artificial\masterIADesarollo\micro-proyecto-grupo-7\.sleep-cv\ajuste-lightgbm\8731d8caf3d24703\modelo_todos_los_sujetos.txt
